In [0]:
# Runs as the pseudonymization service principal (job run_as), so the SP owns every schema it creates.
dbutils.widgets.text("catalog", "serverless_stable_yit814_catalog")
dbutils.widgets.text("sp_application_id", "")
catalog = dbutils.widgets.get("catalog")
RAW, VAULT, GOV, ANALYTICS = (f"{catalog}.pii_raw", f"{catalog}.pii_vault", f"{catalog}.pii_governance", f"{catalog}.pii_analytics")

In [0]:
zones = {
    RAW: "RESTRICTED. Simulated Lakeflow Connect landing (SQL Server), staging, quarantine. Cleartext PII, short retention. Locked down by ABAC for everyone except the pipeline SP.",
    VAULT: "RESTRICTED. UC secrets (AES/HMAC keys), encrypt/decrypt UDFs and token vault tables. Only the pseudonymization SP.",
    GOV: "PII contract (column allow-list + protection method) and batch_control (verification gate results). No PII.",
    ANALYTICS: "PSEUDONYMIZED. Published tables with tokens/ciphertext only. Analysts, data science, BI.",
}
for schema, comment in zones.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema} COMMENT '{comment}'")

# Least privilege. pii_raw gets SELECT so that ABAC (not missing grants) is what hides the data:
# every column is masked and every row filtered for account users. pii_vault gets no grants at all.
spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {RAW} TO `account users`")
spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {GOV} TO `account users`")
spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {ANALYTICS} TO `account users`")
display(spark.sql(f"SHOW SCHEMAS IN {catalog} LIKE 'pii_*'"))

In [0]:
# Every column of every source table must be declared here (allow-list). Columns not in the contract are
# routed to _rescued_data and quarantined by the raw pipeline. method: keep | tokenize | encrypt | drop.
# class_tag uses system governed tags (class.*) because the account's governed-tag quota is full.
spark.sql(f"""
CREATE OR REPLACE TABLE {GOV}.pii_contract (
  table_name STRING, column_name STRING, data_type STRING, pii_type STRING, class_tag STRING,
  method STRING, normalize_rule STRING, key_or_vault STRING, active BOOLEAN)
COMMENT 'PII data contract: drives schema enforcement, tagging, normalization, pseudonymization and verification'
""")
spark.sql(f"""
INSERT INTO {GOV}.pii_contract VALUES
  ('customers','customer_id','BIGINT',NULL,NULL,'keep','none',NULL,true),
  ('customers','full_name','STRING','name','class.name','tokenize','name','tokmap_name',true),
  ('customers','email','STRING','email','class.email_address','tokenize','lower_trim','tokmap_email',true),
  ('customers','phone','STRING','phone','class.phone_number','encrypt','e164','phone_key_v1',true),
  ('customers','ssn','STRING','ssn','class.us_ssn','encrypt','digits_only','ssn_key_v1',true),
  ('customers','dob','DATE','dob','class.date_of_birth','encrypt','none','dob_key_v1',true),
  ('customers','street_address','STRING','address','class.street_address','drop','none',NULL,true),
  ('customers','city','STRING',NULL,NULL,'keep','none',NULL,true),
  ('customers','segment','STRING',NULL,NULL,'keep','none',NULL,true),
  ('customers','signup_date','DATE',NULL,NULL,'keep','none',NULL,true),
  ('transactions','txn_id','STRING',NULL,NULL,'keep','none',NULL,true),
  ('transactions','customer_id','BIGINT',NULL,NULL,'keep','none',NULL,true),
  ('transactions','amount','DECIMAL(10,2)',NULL,NULL,'keep','none',NULL,true),
  ('transactions','currency','STRING',NULL,NULL,'keep','none',NULL,true),
  ('transactions','merchant','STRING',NULL,NULL,'keep','none',NULL,true),
  ('transactions','card_number','STRING','card','class.credit_card','drop','none',NULL,true),
  ('transactions','txn_ts','TIMESTAMP',NULL,NULL,'keep','none',NULL,true)
""")
display(spark.table(f"{GOV}.pii_contract"))